# DS603 — Laboratory 11
## Self-Balancing Trees, Prefix Retrieval, Lossless Compression and String Search



## Imports and conventions
Only standard-library modules are used. AVL heights are edge-based (`height(None) = -1`). Strings are case-sensitive and empty-input behavior follows the handout.

In [1]:
from collections import deque, Counter
import heapq
import random

print("Standard-library imports ready.")

Standard-library imports ready.


# Problem 1 — AVL telemetry store

### Design and invariants
- Each node stores one unique integer key, payload, cached edge-height and child links.
- BST ordering is strict; AVL balance requires `abs(height(left)-height(right)) <= 1` at every node.
- Recursive updates return the new subtree root. Heights are refreshed bottom-up. Rotations update the demoted node before the promoted node.
- Duplicate insertion changes only the payload. Two-child deletion copies both successor key and payload before removing the successor.
- LL/RR require one primitive rotation; LR/RL require two. One insertion has at most two primitive rotations in one rebalancing event; deletion can rebalance multiple ancestors, up to O(log N) primitive rotations.
- Search/insert/delete are O(log N) worst case; cached height O(1); breadth-first profile O(N) time and O(N) queue space.

In [2]:
class AVLNode:
    def __init__(self, key, payload):
        self.key, self.payload, self.height = key, payload, 0
        self.left = self.right = None

class AVLTelemetryStore:
    def __init__(self):
        self.root = None

    @staticmethod
    def _height(node):
        return node.height if node is not None else -1

    def _bf(self, node):
        return self._height(node.left) - self._height(node.right)

    def _update(self, node):
        node.height = 1 + max(self._height(node.left), self._height(node.right))

    def _rotate_right(self, z):
        y = z.left
        assert y is not None
        z.left, y.right = y.right, z
        self._update(z); self._update(y)
        return y

    def _rotate_left(self, z):
        y = z.right
        assert y is not None
        z.right, y.left = y.left, z
        self._update(z); self._update(y)
        return y

    def _rebalance(self, node):
        self._update(node)
        b = self._bf(node)
        if b > 1:
            if self._bf(node.left) < 0:
                node.left = self._rotate_left(node.left)
            return self._rotate_right(node)
        if b < -1:
            if self._bf(node.right) > 0:
                node.right = self._rotate_right(node.right)
            return self._rotate_left(node)
        return node

    def _insert(self, node, key, payload):
        if node is None:
            return AVLNode(key, payload)
        if key < node.key:
            node.left = self._insert(node.left, key, payload)
        elif key > node.key:
            node.right = self._insert(node.right, key, payload)
        else:
            node.payload = payload
            return node
        return self._rebalance(node)

    def insert(self, key, payload):
        self.root = self._insert(self.root, key, payload)

    @staticmethod
    def _minimum(node):
        while node.left is not None:
            node = node.left
        return node

    def _delete(self, node, key):
        if node is None:
            return None, False
        if key < node.key:
            node.left, deleted = self._delete(node.left, key)
        elif key > node.key:
            node.right, deleted = self._delete(node.right, key)
        else:
            deleted = True
            if node.left is None:
                return node.right, True
            if node.right is None:
                return node.left, True
            successor = self._minimum(node.right)
            node.key, node.payload = successor.key, successor.payload
            node.right, _ = self._delete(node.right, successor.key)
        return (self._rebalance(node), True) if deleted else (node, False)

    def delete(self, key):
        self.root, deleted = self._delete(self.root, key)
        return deleted

    def get(self, key):
        node = self.root
        while node:
            if key == node.key:
                return node.payload
            node = node.left if key < node.key else node.right
        return None

    def tree_height(self):
        return self._height(self.root)

    def level_order_profile(self):
        if self.root is None:
            return []
        out, q = [], deque([self.root])
        while q:
            n = q.popleft()
            out.append((n.key, self._bf(n)))
            if n.left: q.append(n.left)
            if n.right: q.append(n.right)
        return out

    def inorder_items(self):
        out, stack, node = [], [], self.root
        while stack or node:
            while node:
                stack.append(node); node = node.left
            node = stack.pop()
            out.append((node.key, node.payload))
            node = node.right
        return out

    def validate(self):
        def walk(n, lo, hi):
            if n is None: return -1, []
            assert lo is None or lo < n.key
            assert hi is None or n.key < hi
            lh, a = walk(n.left, lo, n.key)
            rh, b = walk(n.right, n.key, hi)
            assert n.height == 1 + max(lh, rh)
            assert abs(lh-rh) <= 1
            return n.height, a + [(n.key,n.payload)] + b
        _, items = walk(self.root, None, None)
        assert items == self.inorder_items()
        return True

def run_avl_commands(lines):
    tree, out = AVLTelemetryStore(), []
    for line in lines:
        p = line.split()
        if not p: continue
        if p[0] == "INSERT": tree.insert(int(p[1]), p[2])
        elif p[0] == "DELETE": out.append(f"Deleted: {tree.delete(int(p[1]))}")
        elif p[0] == "GET": out.append(f"Value: {tree.get(int(p[1]))}")
        elif p[0] == "HEIGHT": out.append(f"Height: {tree.tree_height()}")
        elif p[0] == "PROFILE": out.append(f"Profile: {tree.level_order_profile()}")
        else: raise ValueError("Unknown AVL command")
    return out

### Verification
Checks cover LL, RR, LR and RL; duplicate payload update; absent deletion; leaf/one-child/two-child/root deletion; repeated deletion to empty; strict ordering, payload association, cached heights and balance factors.

In [3]:
for seq in ([30,20,10], [10,20,30], [30,10,20], [10,30,20]):
    t = AVLTelemetryStore()
    for k in seq:
        t.insert(k, f"T{k}")
        assert t.validate()
    assert t.root.key == 20 and t.tree_height() == 1

t = AVLTelemetryStore()
for k,v in [(10,"T1"),(20,"T2"),(30,"T3"),(40,"T4"),(50,"T5"),(25,"T2.5")]:
    t.insert(k,v)
assert t.tree_height() == 2
assert t.level_order_profile() == [(30,0),(20,0),(40,-1),(10,0),(25,0),(50,0)]
assert t.delete(30)
assert t.level_order_profile() == [(40,1),(20,0),(50,0),(10,0),(25,0)]
t.insert(25,"UPDATED")
assert t.get(25) == "UPDATED" and not t.delete(999) and t.validate()
for k in [10,50,20,25,40]:
    assert t.delete(k) and t.validate()
assert t.tree_height() == -1 and t.level_order_profile() == []
# One-child and root deletion.
t = AVLTelemetryStore()
for k in [20,10,30,25]: t.insert(k,str(k))
assert t.delete(30) and t.get(25) == "25" and t.validate()
assert t.delete(20) and t.validate()
assert run_avl_commands(["INSERT 10 T1","INSERT 20 T2","INSERT 30 T3","HEIGHT","PROFILE","GET 20","DELETE 10"]) == [
    "Height: 1","Profile: [(20, 0), (10, 0), (30, 0)]","Value: T2","Deleted: True"]
print("AVL verification passed.")

AVL verification passed.


# Problem 2 — Frequency-ranked Trie autocomplete

### Design and invariants
- Each node stores a child map, terminal flag, endpoint frequency and complete word. Shared prefixes share nodes.
- Duplicate insertion adds frequency; distinct-word count changes only on first insertion.
- Prefix lookup follows one edge per character. Suggestions collect endpoint words in the prefix subtree, then sort by decreasing frequency and increasing lexicographic word.
- Deletion clears the endpoint and prunes upward only while a node is nonterminal and childless. This preserves paths shared by other words.
- Insert/search/delete/prefix lookup are O(L) expected time. For suggestions, with V visited subtree nodes and M matching words, collect-and-sort costs O(L + V + M log(M+1)), plus output construction; string comparisons can add character costs. Empty prefix may visit the whole dictionary.

In [4]:
class TrieNode:
    def __init__(self):
        self.children = {}
        self.is_end_of_word = False
        self.frequency = 0
        self.word = None

class AutocompleteTrie:
    def __init__(self):
        self.root, self.word_count = TrieNode(), 0

    @staticmethod
    def _check(word, empty=False):
        if (not empty and not word) or any(not ('a' <= c <= 'z') for c in word):
            raise ValueError("Expected lowercase English word")

    def _find(self, text):
        n = self.root
        for c in text:
            n = n.children.get(c)
            if n is None: return None
        return n

    def insert(self, word, frequency):
        self._check(word)
        if not isinstance(frequency,int) or frequency <= 0:
            raise ValueError("frequency must be positive")
        n = self.root
        for c in word: n = n.children.setdefault(c, TrieNode())
        if not n.is_end_of_word: self.word_count += 1
        n.is_end_of_word = True
        n.frequency += frequency
        n.word = word

    def search(self, word):
        if not word: return False
        self._check(word)
        n = self._find(word)
        return n is not None and n.is_end_of_word

    def starts_with(self, prefix):
        self._check(prefix, empty=True)
        return self.word_count > 0 and self._find(prefix) is not None

    def suggest(self, prefix, k):
        self._check(prefix, empty=True)
        if k < 0: raise ValueError("k must be nonnegative")
        if k == 0: return []
        start = self._find(prefix)
        if start is None: return []
        found, stack = [], [start]
        while stack:
            n = stack.pop()
            if n.is_end_of_word: found.append((n.word,n.frequency))
            stack.extend(n.children.values())
        found.sort(key=lambda x: (-x[1],x[0]))
        return found[:k]

    def delete(self, word):
        if not word: return False
        self._check(word)
        n, path = self.root, []
        for c in word:
            if c not in n.children: return False
            path.append((n,c)); n = n.children[c]
        if not n.is_end_of_word: return False
        n.is_end_of_word, n.frequency, n.word = False, 0, None
        self.word_count -= 1
        for parent,c in reversed(path):
            child = parent.children[c]
            if child.is_end_of_word or child.children: break
            del parent.children[c]
        return True

    def validate(self):
        words = []
        def walk(n, prefix):
            if n.is_end_of_word:
                assert n.word == prefix and n.frequency > 0
                words.append(prefix)
            else:
                assert n.word is None and n.frequency == 0
            for c,ch in n.children.items():
                assert len(c)==1 and 'a'<=c<='z'
                walk(ch,prefix+c)
        walk(self.root,"")
        assert len(words) == self.word_count
        return True

def run_trie_commands(lines):
    trie, out = AutocompleteTrie(), []
    for line in lines:
        p=line.split(); op=p[0]
        if op=="ADD": trie.insert(p[1],int(p[2]))
        elif op=="SEARCH": out.append(f"Search '{p[1]}': {trie.search(p[1])}")
        elif op=="PREFIX":
            x="" if p[1]=='""' else p[1]
            out.append(f"Prefix '{x}': {trie.starts_with(x)}")
        elif op=="SUGGEST":
            x="" if p[1]=='""' else p[1]
            out.append(f"Suggestions for '{x}': {trie.suggest(x,int(p[2]))}")
        elif op=="DELETE": out.append(f"Deleted '{p[1]}': {trie.delete(p[1])}")
        else: raise ValueError("Unknown Trie command")
    return out

### Verification
Checks include the handout sample, additive frequencies, equal-frequency lexical ties, absent prefixes, k=0, oversized k, empty prefix on empty/nonempty Tries, deletion of a prefix word and longer word, reinsertion with fresh frequency and complete deletion.

In [5]:
trie=AutocompleteTrie()
for w,f in [("apple",50),("app",100),("application",30),("apply",40),("banana",90)]:
    trie.insert(w,f)
assert trie.suggest("app",3)==[("app",100),("apple",50),("apply",40)]
assert trie.search("apple") and trie.starts_with("app")
assert trie.delete("app")
assert trie.suggest("app",3)==[("apple",50),("apply",40),("application",30)]
trie.insert("apple",10); trie.insert("ape",60); trie.insert("apt",60)
assert trie.suggest("ap",3)==[("ape",60),("apple",60),("apt",60)]
assert trie.suggest("missing",5)==[] and trie.suggest("ap",0)==[]
assert len(trie.suggest("",100))==trie.word_count and trie.search("") is False
t=AutocompleteTrie()
assert not t.starts_with("") and t.suggest("",10)==[]
t.insert("car",5); t.insert("cart",8); t.insert("care",3)
assert t.delete("car") and t.search("cart") and t.search("care")
assert t.delete("cart") and t.suggest("car",10)==[("care",3)]
assert t.delete("care") and not t.starts_with("car") and not t.starts_with("")
t.insert("car",2)
assert t.suggest("",10)==[("car",2)] and t.validate()
out=run_trie_commands(["ADD apple 50","ADD app 100","ADD application 30","ADD apply 40","ADD banana 90","SUGGEST app 3","SEARCH apple","DELETE app","SUGGEST app 3"])
assert out == ["Suggestions for 'app': [('app', 100), ('apple', 50), ('apply', 40)]","Search 'apple': True","Deleted 'app': True","Suggestions for 'app': [('apple', 50), ('apply', 40), ('application', 30)]"]
print("Trie verification passed.")

Trie verification passed.


# Problem 3 — Deterministic Huffman payload compressor

### Design and invariants
- Count frequencies for printable ASCII (32–126). Insert leaves in ascending ASCII order.
- Heap order is `(frequency, minimum_subtree_character, creation_id)`. First popped node becomes left child; parents receive increasing IDs. Ties are deterministic.
- Symbols occur at leaves, making the code prefix-free. Singleton alphabet uses code `0`; empty input has an empty tree/codebook.
- The bit string is a logical bitstream, not a packed file. Ratio is `B/(8N)` and excludes tree/codebook metadata, byte padding and Python object overhead. It is not an actual memory-size claim.
- Build costs O(N + σ log σ), plus explicit codeword construction; encode O(N+B); decode O(B + output size).

In [6]:
class _HNode:
    def __init__(self, freq, min_char, uid, char=None, left=None, right=None):
        self.freq,self.min_char,self.uid=freq,min_char,uid
        self.char,self.left,self.right=char,left,right
    @property
    def leaf(self): return self.char is not None

class HuffmanPayloadCompressor:
    def __init__(self):
        self.root=None; self.codebook={}; self._initialized=False

    def build_tree(self,text):
        if any(not 32<=ord(c)<=126 for c in text):
            raise ValueError("Only printable ASCII 32–126 is allowed")
        self.root=None; self.codebook={}; self._initialized=True
        if not text: return
        heap=[]; uid=0
        for c in sorted(Counter(text),key=ord):
            n=_HNode(text.count(c),ord(c),uid,char=c)
            heapq.heappush(heap,(n.freq,n.min_char,n.uid,n)); uid+=1
        while len(heap)>1:
            *_,left=heapq.heappop(heap)
            *_,right=heapq.heappop(heap)
            parent=_HNode(left.freq+right.freq,min(left.min_char,right.min_char),uid,left=left,right=right)
            heapq.heappush(heap,(parent.freq,parent.min_char,parent.uid,parent)); uid+=1
        self.root=heap[0][3]
        def visit(n,prefix):
            if n.leaf: self.codebook[n.char]=prefix or "0"
            else: visit(n.left,prefix+"0"); visit(n.right,prefix+"1")
        visit(self.root,"")

    def get_codebook(self): return dict(self.codebook)

    def compress(self,text):
        self.build_tree(text)
        if not text: return "",0,0.0
        bits="".join(self.codebook[c] for c in text)
        return bits,len(bits),len(bits)/(8*len(text))

    def decompress(self,bitstring):
        if not self._initialized: raise ValueError("Build or compress first")
        if any(b not in "01" for b in bitstring): raise ValueError("Nonbinary bitstream")
        if self.root is None:
            if bitstring: raise ValueError("Empty tree accepts only empty stream")
            return ""
        if self.root.leaf:
            if any(b!="0" for b in bitstring): raise ValueError("Singleton accepts only zeros")
            return self.root.char*len(bitstring)
        out=[]; n=self.root
        for b in bitstring:
            n=n.left if b=="0" else n.right
            if n is None: raise ValueError("Nonexistent branch")
            if n.leaf: out.append(n.char); n=self.root
        if n is not self.root: raise ValueError("Truncated internal-node path")
        return "".join(out)

    def is_prefix_free(self):
        codes=list(self.codebook.values())
        return all(a==b or not b.startswith(a) for a in codes for b in codes)

def run_huffman_cases(lines):
    h,out=HuffmanPayloadCompressor(),[]
    for i,s in enumerate(lines,1):
        bits,count,ratio=h.compress(s); raw=8*len(s)
        out.append(f"--- Test {i} ---")
        if s: out.append(f"Raw Bits: {raw} | Compressed Bits: {count} | Ratio: {ratio:.4f} (Savings: {100*(1-ratio):.2f}%)")
        else: out.append("Raw Bits: 0 | Compressed Bits: 0 | Ratio: 0.0000 (Savings: N/A)")
        ordered={c:h.codebook[c] for c in sorted(h.codebook,key=ord)}
        out += [f"Codebook: {ordered}",f"Encoded Bitstream: {bits}",f"Decompressed Match: {h.decompress(bits)==s}"]
    return out

### Verification
Both supplied examples are checked exactly (including codebooks and bitstreams). Further tests cover prefix freedom, weighted bit count, singleton/empty input, space-only input, sequential rebuilds, malformed streams, truncated paths and decoding before initialization.

In [7]:
h=HuffmanPayloadCompressor()
bits,n,r=h.compress("BEEP BOOP BEER!")
assert n==40 and abs(r-1/3)<1e-12
assert h.get_codebook()=={' ':'010','!':'0110','B':'00','E':'10','O':'110','P':'111','R':'0111'}
assert bits=="0010101110100011011011101000101001110110"
assert h.decompress(bits)=="BEEP BOOP BEER!" and h.is_prefix_free()
assert n==sum(Counter("BEEP BOOP BEER!")[c]*len(code) for c,code in h.codebook.items())
bits,n,r=h.compress("AAAAAAABBBCCCCDD")
assert n==30 and abs(r-30/128)<1e-12
assert h.get_codebook()=={'A':'0','B':'111','C':'10','D':'110'}
assert bits=="000000011111111110101010110110" and h.decompress(bits)=="AAAAAAABBBCCCCDD"
assert h.compress("ZZZZ")==("0000",4,0.125) and h.decompress("000")=="ZZZ"
assert h.compress("   ")==("000",3,0.125)
assert h.compress("")==("",0,0.0) and h.get_codebook()=={} and h.decompress("")==""
fresh=HuffmanPayloadCompressor()
try: fresh.decompress(""); assert False
except ValueError: pass
h.compress("AAABBC")
for bad in ["2","01x"]:
    try: h.decompress(bad); assert False, "nonbinary stream accepted"
    except ValueError: pass
truncated = next(code[:-1] for code in h.codebook.values() if len(code) > 1)
try: h.decompress(truncated); assert False, "truncated internal-node path accepted"
except ValueError: pass
h.compress("QQ")
try: h.decompress("1"); assert False
except ValueError: pass
try: h.compress("bad\ntext"); assert False
except ValueError: pass
a,b=HuffmanPayloadCompressor(),HuffmanPayloadCompressor()
a.build_tree("equal frequencies!"); b.build_tree("equal frequencies!")
assert a.get_codebook()==b.get_codebook()
print("Huffman verification passed.")
print("\n".join(run_huffman_cases(["BEEP BOOP BEER!","AAAAAAABBBCCCCDD"])[:3]))

Huffman verification passed.
--- Test 1 ---
Raw Bits: 120 | Compressed Bits: 40 | Ratio: 0.3333 (Savings: 66.67%)
Codebook: {' ': '010', '!': '0110', 'B': '00', 'E': '10', 'O': '110', 'P': '111', 'R': '0111'}


### Viva preparation
1. Huffman is optimal among binary symbol-by-symbol prefix-free codes for the given frequencies. For at least two positive-probability symbols, `H <= average code length < H+1`. Metadata and padding can outweigh savings for a small message.
2. An empty singleton codeword emits no bits regardless of repetition count; without a separate count, the decoder cannot know how many copies to produce. Code `0` preserves the repetition count.

# Problem 4 — KMP sequence scanner and comparison benchmark

### Design and invariants
- `lps[i]` is the longest proper prefix of `pattern[:i+1]` that is also its suffix.
- After j matched characters, `text[i-j:i] == pattern[:j]`. On mismatch, fallback to `lps[j-1]` without moving i; this retains a valid suffix and avoids rescanning text.
- After a complete match, fallback also preserves overlaps. Empty pattern convention: no matches and `(0,0)` comparisons.
- LPS is O(M), search O(N) plus O(r) returned-position storage.
- Benchmark counts only character comparisons in search loops, not LPS preprocessing, integer comparisons or loop checks. Naive tests every legal window left-to-right. KMP uses the reference `i < len(text)` loop without early-stop optimization.

In [8]:
class GenomicKMPScanner:
    @staticmethod
    def compute_lps(pattern):
        lps=[0]*len(pattern); length=0; i=1
        while i<len(pattern):
            if pattern[i]==pattern[length]:
                length+=1; lps[i]=length; i+=1
            elif length: length=lps[length-1]
            else: i+=1
        return lps

    def search(self,text,pattern):
        if not pattern: return []
        lps=self.compute_lps(pattern); out=[]; i=j=0
        while i<len(text):
            if text[i]==pattern[j]:
                i+=1; j+=1
                if j==len(pattern):
                    out.append(i-j); j=lps[j-1]
            elif j: j=lps[j-1]
            else: i+=1
        return out

    def benchmark_comparisons(self,text,pattern):
        if not pattern: return (0,0)
        naive=0
        for start in range(len(text)-len(pattern)+1):
            j=0
            while j<len(pattern):
                naive+=1
                if text[start+j]!=pattern[j]: break
                j+=1
        lps=self.compute_lps(pattern); kmp=0; i=j=0
        while i<len(text):
            kmp+=1
            if text[i]==pattern[j]:
                i+=1; j+=1
                if j==len(pattern): j=lps[j-1]
            elif j: j=lps[j-1]
            else: i+=1
        return naive,kmp

def _empty_token(s): return "" if s=='""' else s

def run_kmp_commands(lines):
    k,out=GenomicKMPScanner(),[]
    for line in lines:
        p=line.split()
        if p[0]=="LPS": out.append(f"LPS: {k.compute_lps(_empty_token(p[1]))}")
        elif p[0]=="FIND":
            x,y=map(_empty_token,p[1:3]); out.append(f"Occurrences: {k.search(x,y)}")
        elif p[0]=="BENCH":
            x,y=map(_empty_token,p[1:3]); a,b=k.benchmark_comparisons(x,y)
            saving="N/A" if a==0 else f"{100*(a-b)/a:.2f}%"
            out.append(f"Benchmark: Naive Comparisons: {a} | KMP Comparisons: {b} | Savings: {saving}")
        else: raise ValueError("Unknown KMP command")
    return out

### Verification
Checks include both LPS examples, overlaps, no/full match, longer pattern, empty inputs, one-character pattern, late mismatch, benchmark sample and randomized comparison with an all-windows reference matcher (used only for testing).

In [9]:
k=GenomicKMPScanner()
assert k.compute_lps("AABAACAABAA")==[0,1,0,1,2,0,1,2,3,4,5]
assert k.compute_lps("GATCAGAT")==[0,0,0,0,0,1,2,3]
assert k.search("GATCAGATCAGAT","GATCA")==[0,5]
assert k.search("aaaaa","aaa")==[0,1,2]
assert k.search("abcdef","xyz")==[] and k.search("abc","abc")==[0]
assert k.search("ab","abcd")==[] and k.search("","a")==[] and k.search("abc","")==[]
assert k.compute_lps("")==[] and k.benchmark_comparisons("anything","")== (0,0)
assert k.benchmark_comparisons("A"*20,"A"*5)==(80,20)
assert k.search("A"*1000,"AAAAB")==[]
def naive_ref(text,pat):
    if not pat: return []
    return [i for i in range(len(text)-len(pat)+1) if text[i:i+len(pat)]==pat]
rng=random.Random(603)
for _ in range(250):
    text="".join(rng.choice("ABCD") for _ in range(rng.randrange(35)))
    pat="".join(rng.choice("ABCD") for _ in range(rng.randrange(9)))
    assert k.search(text,pat)==naive_ref(text,pat)
cmd=run_kmp_commands(["LPS AABAACAABAA","LPS GATCAGAT","FIND GATCAGATCAGAT GATCA","BENCH AAAAAAAAAAAAAAAAAAAA AAAAA"])
assert cmd[-1]=="Benchmark: Naive Comparisons: 80 | KMP Comparisons: 20 | Savings: 75.00%"
print("KMP verification passed.")
print("\n".join(cmd))

KMP verification passed.
LPS: [0, 1, 0, 1, 2, 0, 1, 2, 3, 4, 5]
LPS: [0, 0, 0, 0, 0, 1, 2, 3]
Occurrences: [0, 5]
Benchmark: Naive Comparisons: 80 | KMP Comparisons: 20 | Savings: 75.00%


# Problem 5 — Weighted sequence alignment engine

### Design and invariants
- `D[i][j]` is the minimum cost to transform source prefix of length i into target prefix of length j.
- Boundaries: `D[i][0]=i*del_cost`; `D[0][j]=j*ins_cost`. Interior cells minimize diagonal match/replace, delete-from-source and insert-target.
- Traceback considers only eligible predecessors and prefers diagonal, then delete, then insert. Reverse collected operations for a left-to-right script.
- Rolling rows orient the shorter string along columns. If source/target are swapped, insertion and deletion costs must also swap.
- Full table is O(MN) time and O((M+1)(N+1)) space. Rolling rows are O(MN) time and O(min(M,N)+1) auxiliary space. Last two rows alone do not preserve traceback history; Hirschberg's divide-and-conquer is a linear-space reconstruction alternative.

In [10]:
class SequenceDiffEngine:
    @staticmethod
    def _costs(ins,dele,sub):
        if any(not isinstance(x,int) or x<=0 for x in (ins,dele,sub)):
            raise ValueError("Costs must be positive integers")

    def compute_distance_matrix(self,source,target,ins_cost=1,del_cost=1,sub_cost=1):
        self._costs(ins_cost,del_cost,sub_cost)
        m,n=len(source),len(target)
        dp=[[0]*(n+1) for _ in range(m+1)]
        for i in range(1,m+1): dp[i][0]=i*del_cost
        for j in range(1,n+1): dp[0][j]=j*ins_cost
        for i in range(1,m+1):
            for j in range(1,n+1):
                delta=0 if source[i-1]==target[j-1] else sub_cost
                dp[i][j]=min(dp[i-1][j-1]+delta,dp[i-1][j]+del_cost,dp[i][j-1]+ins_cost)
        return dp[m][n],dp

    def reconstruct_alignment(self,source,target,dp_matrix,ins_cost=1,del_cost=1,sub_cost=1):
        self._costs(ins_cost,del_cost,sub_cost)
        m,n=len(source),len(target)
        if len(dp_matrix)!=m+1 or any(len(row)!=n+1 for row in dp_matrix):
            raise ValueError("DP matrix dimensions mismatch")
        i,j=m,n; rev=[]
        while i or j:
            cur=dp_matrix[i][j]
            if i and j:
                delta=0 if source[i-1]==target[j-1] else sub_cost
                if dp_matrix[i-1][j-1]+delta==cur:
                    rev.append(f"MATCH {source[i-1]}" if delta==0 else f"REPLACE {source[i-1]} -> {target[j-1]}")
                    i-=1; j-=1; continue
            if i and dp_matrix[i-1][j]+del_cost==cur:
                rev.append(f"DELETE {source[i-1]}"); i-=1; continue
            if j and dp_matrix[i][j-1]+ins_cost==cur:
                rev.append(f"INSERT {target[j-1]}"); j-=1; continue
            raise ValueError("Inconsistent DP matrix or costs")
        return rev[::-1]

    def memory_optimized_distance(self,source,target,ins_cost=1,del_cost=1,sub_cost=1):
        self._costs(ins_cost,del_cost,sub_cost)
        if len(target)>len(source):
            source,target=target,source
            ins_cost,del_cost=del_cost,ins_cost
        n=len(target); prev=[j*ins_cost for j in range(n+1)]
        for i,a in enumerate(source,1):
            cur=[i*del_cost]+[0]*n
            for j,b in enumerate(target,1):
                delta=0 if a==b else sub_cost
                cur[j]=min(prev[j-1]+delta,prev[j]+del_cost,cur[j-1]+ins_cost)
            prev=cur
        return prev[n]

    def verify_script(self,source,target,operations,ins_cost=1,del_cost=1,sub_cost=1):
        i,out,total=0,[],0
        for op in operations:
            if op.startswith("MATCH "):
                c=op[6:]; assert i<len(source) and source[i]==c
                out.append(c); i+=1
            elif op.startswith("REPLACE "):
                a,b=op[8:].split(" -> "); assert i<len(source) and source[i]==a
                out.append(b); i+=1; total+=sub_cost
            elif op.startswith("DELETE "):
                c=op[7:]; assert i<len(source) and source[i]==c
                i+=1; total+=del_cost
            elif op.startswith("INSERT "):
                c=op[7:]; out.append(c); total+=ins_cost
            else: raise AssertionError("Unknown edit operation")
        assert i==len(source) and "".join(out)==target
        return total

def run_edit_distance_cases(cases):
    e,out=SequenceDiffEngine(),[]
    for no,(src,dst,ins,dele,sub) in enumerate(cases,1):
        cost,dp=e.compute_distance_matrix(src,dst,ins,dele,sub)
        roll=e.memory_optimized_distance(src,dst,ins,dele,sub)
        script=e.reconstruct_alignment(src,dst,dp,ins,dele,sub)
        out += [f"--- Pair {no}: {src} -> {dst} ---",f"Minimum Edit Cost: {cost}",
                f"Memory-Optimized Cost: {roll} (Matches full table!)" if roll==cost else f"Memory-Optimized Cost: {roll} (MISMATCH)",
                "Optimal Transformation Script:"]
        out += [f"{i}. {op}" for i,op in enumerate(script,1)] or ["(empty script)"]
    return out

### Verification
Checks cover both supplied pairs, identical strings, empty source/target, replacement, repeated-character ties, substitution equal to/greater than insertion plus deletion, asymmetric costs, rolling-row equality and script replay (source consumed, target produced, penalties summed).

In [11]:
e=SequenceDiffEngine()
def check(src,dst,ins=1,dele=1,sub=1):
    cost,dp=e.compute_distance_matrix(src,dst,ins,dele,sub)
    roll=e.memory_optimized_distance(src,dst,ins,dele,sub)
    script=e.reconstruct_alignment(src,dst,dp,ins,dele,sub)
    replay=e.verify_script(src,dst,script,ins,dele,sub)
    assert cost==roll==replay
    return cost,script,dp
assert check("INTENTION","EXECUTION")[0]==5
assert check("HORSE","ROS")[0]==3
assert check("same","same")[0]==0
assert check("","")[0:2]==(0,[])
assert check("","abc",2,3,4)[0]==6 and check("abc","",2,3,4)[0]==9
assert check("A","B",1,1,1)[0]==1
assert check("A","B",1,1,2)[1]==["REPLACE A -> B"]
assert check("A","B",1,1,3)[1]==["INSERT B","DELETE A"]
assert check("A","",2,5,1)[0]==5 and check("","A",2,5,1)[0]==2
check("AAAB","ABAA",2,3,2)
for src in ["","A","B","AA","AB","BA"]:
    for dst in ["","A","B","AA","AB","BA"]:
        for costs in [(1,1,1),(2,5,1),(1,3,4),(3,1,2)]:
            check(src,dst,*costs)
sample=run_edit_distance_cases([("INTENTION","EXECUTION",1,1,1),("HORSE","ROS",1,1,1)])
assert sample[1]=="Minimum Edit Cost: 5" and sample[2]=="Memory-Optimized Cost: 5 (Matches full table!)"
print("Edit-distance verification passed.")
print("\n".join(sample))

Edit-distance verification passed.
--- Pair 1: INTENTION -> EXECUTION ---
Minimum Edit Cost: 5
Memory-Optimized Cost: 5 (Matches full table!)
Optimal Transformation Script:
1. REPLACE I -> E
2. REPLACE N -> X
3. REPLACE T -> E
4. REPLACE E -> C
5. REPLACE N -> U
6. MATCH T
7. MATCH I
8. MATCH O
9. MATCH N
--- Pair 2: HORSE -> ROS ---
Minimum Edit Cost: 3
Memory-Optimized Cost: 3 (Matches full table!)
Optimal Transformation Script:
1. REPLACE H -> R
2. MATCH O
3. DELETE R
4. MATCH S
5. DELETE E
